In [5]:
!pip install -r requirements.txt

In [6]:
import pandas as pd
import os
import librosa
import numpy as np
import tarfile
import glob
from joblib import Parallel, delayed
from tqdm import tqdm
from tqdm_joblib import tqdm_joblib
import joblib

# Model, Scaler, and Eval
import lightgbm as lgb
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report


BASE_PATH = "ASVspoof5/"
PROTOCOL_PATH = BASE_PATH + "ASVspoof5_protocols/"

In [7]:
# Protocol Extraction

os.makedirs(PROTOCOL_PATH, exist_ok=True)

PROTOCOL_TAR = "ASVspoof5_protocols.tar.gz"

with tarfile.open(BASE_PATH+PROTOCOL_TAR, "r:gz") as prot:
    prot.extractall(PROTOCOL_PATH)


C:\Users\dynor\AppData\Local\Temp\ipykernel_23692\4242378183.py:8: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  prot.extractall(PROTOCOL_PATH)


In [8]:
# Flac Extraction

def extract_flac(flac_path):

    flac_name = os.path.basename(flac_path)
    name_parts = flac_name.split("_")

    if len(name_parts) < 3:
        print(f"Unknown file: {flac_name}")
        return
    
    try:
        with tarfile.open(flac_path, "r") as flac:
            flac.extractall(BASE_PATH)
            print(f"Done {flac_name}")
            return flac_name, "Success"
    except Exception as e:
        print(f"Failure in extraction: {e}")


# Find all flac_*.tar files
flac_files = sorted(glob.glob(BASE_PATH + "flac_*.tar"))

n_jobs = max(1, os.cpu_count() - 2) # cores available - 2

with tqdm_joblib(tqdm(total=len(flac_files), desc="Extracting tar files")) as progress_bar:
    results = Parallel(n_jobs=n_jobs)(
        delayed(extract_flac)(f) for f in flac_files
    )
# Result Check
for flac_name, status in results:
    print(f"{flac_name}: {status}")

  0%|          | 0/3 [00:00<?, ?it/s]

flac_D_aa.tar: Success
flac_E_aa.tar: Success
flac_T_aa.tar: Success


In [9]:
# Loads Protocol DataFrames
columns = ["SPEAKER_ID","FLAC_FILE_NAME","SPEAKER_GENDER","CODEC","CODEC_Q","CODEC_SEED","ATTACK_TAG","ATTACK_LABEL","KEY","TMP"]

def load_asvspoof_tsv(tsv_path: str) -> pd.DataFrame:
    df = pd.read_csv(
        tsv_path,
        sep=r'\s+',
        names=columns,
        engine="python",
        dtype=str,
    )
    return df

train_df = load_asvspoof_tsv(PROTOCOL_PATH+"ASVspoof5.train.tsv")
dev_df = load_asvspoof_tsv(PROTOCOL_PATH+"ASVspoof5.dev.track_1.tsv")
eval_df = load_asvspoof_tsv(PROTOCOL_PATH+"ASVspoof5.eval.track_1.tsv")

Extracting tar files:   0%|          | 0/3 [02:28<?, ?it/s]


In [10]:
print(train_df.head())

  SPEAKER_ID FLAC_FILE_NAME SPEAKER_GENDER CODEC CODEC_Q CODEC_SEED  \
0     T_4850   T_0000000000              F     -       -          -   
1     T_0858   T_0000000001              M     -       -          -   
2     T_4075   T_0000000002              M     -       -          -   
3     T_0938   T_0000000003              M     -       -          -   
4     T_0610   T_0000000004              M     -       -          -   

  ATTACK_TAG ATTACK_LABEL    KEY TMP  
0        AC3          A05  spoof   -  
1        AC3          A07  spoof   -  
2        AC2          A04  spoof   -  
3        AC2          A08  spoof   -  
4        AC2          A05  spoof   -  


In [11]:
# Adds Flac File Path to Protocol DataFrames
TEST_PATH = BASE_PATH + "flac_T/"
DEV_PATH = BASE_PATH + "flac_D/"
EVAL_PATH = BASE_PATH + "flac_E_eval/"

def audio_Tfile_check(file_name:str) -> str:
    if not file_name.startswith("T_"):
        raise ValueError(F"Unrecognized Tfile, {file_name}")
    return TEST_PATH+file_name+".flac"

def audio_Dfile_check(file_name:str) -> str:
    if not file_name.startswith("D_"):
        raise ValueError(F"Unrecognized Dfile, {file_name}")
    return DEV_PATH+file_name+".flac"

def audio_Efile_check(file_name:str) -> str:
    if not file_name.startswith("E_"):
        raise ValueError(F"Unrecognized Efile, {file_name}")
    return EVAL_PATH+file_name+".flac"

train_df["FULL_FILE_PATH"] = train_df["FLAC_FILE_NAME"].apply(audio_Tfile_check)
dev_df["FULL_FILE_PATH"] = dev_df["FLAC_FILE_NAME"].apply(audio_Dfile_check)
eval_df["FULL_FILE_PATH"] = eval_df["FLAC_FILE_NAME"].apply(audio_Efile_check)

In [12]:
print(eval_df.head())

  SPEAKER_ID FLAC_FILE_NAME SPEAKER_GENDER CODEC CODEC_Q    CODEC_SEED  \
0     E_1607   E_0009538969              M   C05       2  E_0009486171   
1     E_3614   E_0009249178              F   C11       4  E_0006756001   
2     E_2192   E_0004993854              F     -       0             -   
3     E_0237   E_0006624752              M   C01       1  E_0007675046   
4     E_0884   E_0007708293              F   C07       5  E_0006994972   

  ATTACK_TAG ATTACK_LABEL       KEY TMP  \
0        AC1          A26     spoof   -   
1        AC1          A27     spoof   -   
2          -     bonafide  bonafide   -   
3        AC2          A25     spoof   -   
4        AC1          A21     spoof   -   

                            FULL_FILE_PATH  
0  ASVspoof5/flac_E_eval/E_0009538969.flac  
1  ASVspoof5/flac_E_eval/E_0009249178.flac  
2  ASVspoof5/flac_E_eval/E_0004993854.flac  
3  ASVspoof5/flac_E_eval/E_0006624752.flac  
4  ASVspoof5/flac_E_eval/E_0007708293.flac  


In [13]:
# TEST ONLY! DELETE FOR FULL DATASET
# Filter Protocol Dataframes to available files

existing_mask = train_df["FULL_FILE_PATH"].apply(os.path.isfile)
available_train_df = train_df[existing_mask].copy().reset_index(drop=True)

existing_mask = dev_df["FULL_FILE_PATH"].apply(os.path.isfile)
available_dev_df = dev_df[existing_mask].copy().reset_index(drop=True)

existing_mask = eval_df["FULL_FILE_PATH"].apply(os.path.isfile)
available_eval_df = eval_df[existing_mask].copy().reset_index(drop=True)

In [14]:
# TEST ONLY! DELETE FOR FULL DATASET

print(f"Files actually found on disk: {len(available_train_df):,} / {len(train_df):,}")
print(f"→ Using only these {len(available_train_df):,} utterances for now.\n")
print(available_dev_df.value_counts())
print(available_eval_df.value_counts())


Files actually found on disk: 36,500 / 182,357
→ Using only these 36,500 utterances for now.

SPEAKER_ID  FLAC_FILE_NAME  SPEAKER_GENDER  CODEC  CODEC_Q  CODEC_SEED  ATTACK_TAG  ATTACK_LABEL  KEY       TMP  FULL_FILE_PATH                    
D_0003      D_0000001555    F               -      -        -           -           bonafide      bonafide  -    ASVspoof5/flac_D/D_0000001555.flac    1
            D_0000052312    F               -      -        -           -           bonafide      bonafide  -    ASVspoof5/flac_D/D_0000052312.flac    1
            D_0000110482    F               -      -        -           -           bonafide      bonafide  -    ASVspoof5/flac_D/D_0000110482.flac    1
            D_0000395851    F               -      -        -           -           bonafide      bonafide  -    ASVspoof5/flac_D/D_0000395851.flac    1
            D_0000487852    F               -      -        -           -           bonafide      bonafide  -    ASVspoof5/flac_D/D_0000487852.fla

In [15]:
# TEST ONLY! DELETE FOR FULL DATASET
# Reduction for Test
N_BON = 1200
N_SPOOF = 1800 
def set_reduction(df) -> pd.DataFrame:
    bon_df = df[df["ATTACK_LABEL"] == "bonafide"]
    spoof_df = df[df["ATTACK_LABEL"] != "bonafide"]

    n_bon_real   = min(N_BON,   len(bon_df))
    n_spoof_real = min(N_SPOOF, len(spoof_df))

    print(f"Taking {n_bon_real:,} bonafide + {n_spoof_real:,} spoof samples")

    small_df = pd.concat([
        bon_df.sample(n=n_bon_real,   random_state=42),
        spoof_df.sample(n=n_spoof_real, random_state=42)
    ]).sample(frac=1, random_state=43).reset_index(drop=True) 
    return small_df

small_train_df = set_reduction(available_train_df)
small_dev_df = set_reduction(available_dev_df)
small_eval_df = set_reduction(available_eval_df)

Taking 1,200 bonafide + 1,800 spoof samples
Taking 1,200 bonafide + 1,800 spoof samples
Taking 1,200 bonafide + 1,800 spoof samples


In [16]:
# TEST ONLY! DELETE FOR FULL DATASET
print(small_train_df.value_counts())

SPEAKER_ID  FLAC_FILE_NAME  SPEAKER_GENDER  CODEC  CODEC_Q  CODEC_SEED  ATTACK_TAG  ATTACK_LABEL  KEY       TMP  FULL_FILE_PATH                    
T_0012      T_0000000507    M               -      -        -           -           bonafide      bonafide  -    ASVspoof5/flac_T/T_0000000507.flac    1
            T_0000001817    M               -      -        -           -           bonafide      bonafide  -    ASVspoof5/flac_T/T_0000001817.flac    1
            T_0000006061    M               -      -        -           -           bonafide      bonafide  -    ASVspoof5/flac_T/T_0000006061.flac    1
            T_0000009444    M               -      -        -           -           bonafide      bonafide  -    ASVspoof5/flac_T/T_0000009444.flac    1
            T_0000010717    M               -      -        -           -           bonafide      bonafide  -    ASVspoof5/flac_T/T_0000010717.flac    1
                                                                                       

In [17]:
# Feature Extraction Function
def extract_prosodic_features(flac_path):
    y, sr = librosa.load(flac_path, sr=None)
    
    # --- Pitch (F0) ---    pitch detection pyin
    f0, voiced_flag, voiced_probs = librosa.pyin(
        y, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7')
    )
    f0_voiced = f0[voiced_flag]
    
    pitch_feats = {
        "f0_mean":   np.nanmean(f0_voiced) if len(f0_voiced) else 0,
        "f0_std":    np.nanstd(f0_voiced)  if len(f0_voiced) else 0,
        "f0_range":  np.nanmax(f0_voiced) - np.nanmin(f0_voiced) if len(f0_voiced) else 0,
        "f0_slope":  np.polyfit(np.arange(len(f0_voiced)), f0_voiced, 1)[0] if len(f0_voiced) > 1 else 0,
    }
    
    # --- Energy (RMS) ---
    rms = librosa.feature.rms(y=y)[0]
    energy_feats = {
        "rms_mean": np.mean(rms),
        "rms_std":  np.std(rms),
        "rms_max":  np.max(rms),
    }
    
    # --- Speaking Rate ---
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    rate_feats = {
        "zcr_mean": np.mean(zcr),
        "zcr_std":  np.std(zcr),
    }
    
    # --- Rhythm / Tempo ---
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    tempo = float(np.squeeze(tempo))
    
    # --- Voiced/Unvoiced ratio ---
    voiced_ratio = np.sum(voiced_flag) / len(voiced_flag) if len(voiced_flag) else 0
    
    return {**pitch_feats, **energy_feats, **rate_feats, 
            "tempo": tempo,
            "voiced_ratio": voiced_ratio}

In [18]:
# Feature Caching/Loading

CACHE_DIR = BASE_PATH + "feature_cache/"
os.makedirs(CACHE_DIR, exist_ok=True)


def get_features(path):
    # safe filename for cache
    safe_name = path.replace("/", "_").replace(":", "_")
    cache_path = CACHE_DIR + safe_name + ".pkl"

    # load if cached
    if os.path.exists(cache_path):
        return joblib.load(cache_path)

    # compute features
    feats = extract_prosodic_features(path)

    # save to cache
    joblib.dump(feats, cache_path)
    return feats


def process_row(row):
    feats = get_features(row["FULL_FILE_PATH"])
    feats["ATTACK_LABEL"] = row["ATTACK_LABEL"]
    feats["SPEAKER_GENDER"] = row["SPEAKER_GENDER"]
    feats["SPEAKER_ID"] = row["SPEAKER_ID"]
    return feats



In [19]:
# Build Feature Dataframes

def build_feature_df(df, n_jobs):
    df_list = df.to_dict(orient="records")

    with tqdm_joblib(tqdm(desc="Processing files", total=len(df_list))):
        records = Parallel(n_jobs=n_jobs)(
            delayed(process_row)(row) for row in df_list
        )

    return pd.DataFrame(records)
# n_jobs calculated during flac extraction
# train_feats = build_feature_df(train_df, n_jobs)
# dev_feats   = build_feature_df(dev_df, n_jobs)
# eval_feats  = build_feature_df(eval_df, n_jobs)

# TEST ONLY! DELETE FOR FULL DATASET
train_feats = build_feature_df(small_train_df, n_jobs)
dev_feats   = build_feature_df(small_dev_df, n_jobs)
eval_feats  = build_feature_df(small_eval_df, n_jobs)

Processing files:   0%|          | 0/3000 [00:00<?, ?it/s]

  0%|          | 0/3000 [00:00<?, ?it/s]

  0%|          | 0/3000 [00:00<?, ?it/s]

  0%|          | 0/3000 [00:00<?, ?it/s]

In [20]:
# Simple Labeling for Attack/Bonafide
def map_label(label):
    if label == "bonafide":
        return 0
    else:
        return 1   # any A** is attack

train_feats["LABEL"] = train_feats["ATTACK_LABEL"].apply(map_label)
dev_feats["LABEL"]   = dev_feats["ATTACK_LABEL"].apply(map_label)
eval_feats["LABEL"]  = eval_feats["ATTACK_LABEL"].apply(map_label)

In [21]:
print(train_feats.head())

      f0_mean     f0_std    f0_range  f0_slope  rms_mean   rms_std   rms_max  \
0  285.429504  20.472596  123.726439  0.003062  0.029398  0.013616  0.069589   
1  160.283511  26.605311  130.099103 -0.042898  0.025452  0.015097  0.062207   
2  154.431735  12.437531   93.386967 -0.025455  0.045548  0.022448  0.099537   
3  136.866664  12.116346   64.347507 -0.026583  0.019235  0.013437  0.067621   
4  140.794413  13.510689   82.115510 -0.005937  0.031937  0.016726  0.084625   

   zcr_mean   zcr_std       tempo  voiced_ratio ATTACK_LABEL SPEAKER_GENDER  \
0  0.179325  0.120342  125.000000      0.819572          A01              F   
1  0.180810  0.131123  170.454545      0.879888          A07              M   
2  0.123237  0.107476  170.454545      0.878378          A03              M   
3  0.172878  0.130059   89.285714      0.533007          A03              M   
4  0.132986  0.143816  144.230769      0.836788          A02              M   

  SPEAKER_ID  LABEL  
0     T_3996      1  


In [22]:

FEATURE_COLS = [
    "f0_mean", "f0_std", "f0_range", "f0_slope",
    "rms_mean", "rms_std", "rms_max",
    "zcr_mean", "zcr_std",
    "tempo", "voiced_ratio"
]

# Get x, y values
X_train = train_feats[FEATURE_COLS].values
y_train = train_feats["LABEL"].values

X_dev   = dev_feats[FEATURE_COLS].values
y_dev   = dev_feats["LABEL"].values

X_eval  = eval_feats[FEATURE_COLS].values
y_eval  = eval_feats["LABEL"].values

# Scale
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_dev   = scaler.transform(X_dev)
X_eval  = scaler.transform(X_eval)

# Train
model = lgb.LGBMClassifier(
    n_estimators=1000,
    max_depth=4,
    learning_rate=0.05,
    num_leaves=31,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_dev, y_dev)],
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(period=50)]
)

[LightGBM] [Info] Number of positive: 1800, number of negative: 1200
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000375 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2572
[LightGBM] [Info] Number of data points in the train set: 3000, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.600000 -> initscore=0.405465
[LightGBM] [Info] Start training from score 0.405465
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 50 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] 

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,4
,learning_rate,0.05
,n_estimators,1000
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [23]:

# Final evaluation
y_pred = model.predict(X_eval)
y_prob = model.predict_proba(X_eval)[:, 1]

print(classification_report(y_eval, y_pred))
print("AUC-ROC:", roc_auc_score(y_eval, y_prob))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00      1200
           1       0.60      1.00      0.75      1800

    accuracy                           0.60      3000
   macro avg       0.30      0.50      0.38      3000
weighted avg       0.36      0.60      0.45      3000

AUC-ROC: 0.5863736111111112


c:\Users\dynor\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\dynor\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\dynor\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\dynor\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with n

In [24]:
# Model Saving
joblib.dump(model, 'Trained-Prosodic-Model.joblib')

['Trained-Prosodic-Model.joblib']

In [25]:
# Model Loading
model1 = joblib.load("Trained-Prosodic-Model.joblib")

# Ready to predict
y_pred1 = model1.predict(X_eval)
y_prob1 = model1.predict_proba(X_eval)[:, 1]

print(classification_report(y_eval, y_pred1))
print("AUC-ROC:", roc_auc_score(y_eval, y_prob1))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00      1200
           1       0.60      1.00      0.75      1800

    accuracy                           0.60      3000
   macro avg       0.30      0.50      0.38      3000
weighted avg       0.36      0.60      0.45      3000

AUC-ROC: 0.5863736111111112


c:\Users\dynor\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\dynor\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\dynor\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\dynor\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with n